# Predicting Stellar Class — GPU-стек (Polars + RAPIDS)

Учебный ноутбук: тот же пайплайн, что в `solution_v2.ipynb`, но на GPU-библиотеках.

## Стек
| Слой | Библиотека | Роль |
|------|------------|------|
| ETL / FE | **Polars** | быстрая загрузка, lazy-пайплайн, feature engineering |
| DataFrame на GPU | **cuDF** (RAPIDS) | describe, corr, фильтрации |
| ML на GPU | **XGBoost GPU** + **cuML** | gradient boosting + Random Forest |
| Метрики / CV-сплиты | sklearn | StratifiedKFold, balanced_accuracy |

## Установка на Windows + RTX 4060

**RAPIDS (cuDF/cuML) нативно не работает на Windows** — только через **WSL2 + Ubuntu**.

### Вариант A — полный GPU-стек (рекомендуется)

1. [WSL2 + Ubuntu 22.04](https://learn.microsoft.com/ru-ru/windows/wsl/install)
2. Драйвер NVIDIA с поддержкой WSL (Game Ready / Studio, CUDA WSL)
3. В WSL:

```bash
# conda
wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
bash Miniconda3-latest-Linux-x86_64.sh

# RAPIDS 24.12 + CUDA 12 (RTX 4060 — compute capability 8.9)
conda create -n rapids -c rapidsai -c conda-forge -c nvidia \
    rapids=24.12 python=3.11 cuda-version=12.0
conda activate rapids

# Polars + Jupyter
pip install polars jupyter ipywidgets
```

4. Jupyter в WSL, открыть ноутбук из папки проекта (через `\\wsl$\...` или клон в WSL).

### Вариант B — только Windows (без RAPIDS)

Polars + XGBoost/LightGBM на GPU работают нативно. cuDF/cuML будут недоступны — ноутбук переключится в fallback-режим.

```powershell
pip install polars xgboost lightgbm cupy-cuda12x jupyter
```

> На маленьком датасете GPU может быть не быстрее CPU — цель ноутбука: **научиться API**, а не выжать FPS.

## 0. Проверка окружения

In [ ]:
import os
import ast
import time
import platform
import subprocess
from datetime import datetime
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, classification_report

import xgboost as xgb
from xgboost import XGBClassifier

from IPython.display import display

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100
%config InlineBackend.figure_format = 'retina'


def log(msg):
    print(f"[{datetime.now():%H:%M:%S}] {msg}")


def check_gpu_stack():
    """Проверяет доступность GPU-стека и возвращает флаги."""
    info = {
        'nvidia_smi': False,
        'cudf': False,
        'cuml': False,
        'cupy': False,
        'xgboost_gpu': False,
        'lightgbm_gpu': False,
    }

    try:
        out = subprocess.check_output(['nvidia-smi', '-L'], stderr=subprocess.STDOUT, text=True)
        info['nvidia_smi'] = True
        log(f'GPU: {out.strip()}')
    except Exception as e:
        log(f'nvidia-smi недоступен: {e}')

    try:
        import cudf  # noqa: F401
        info['cudf'] = True
    except ImportError:
        log('cuDF не установлен (нужен RAPIDS в WSL2)')

    try:
        import cuml  # noqa: F401
        info['cuml'] = True
    except ImportError:
        log('cuML не установлен')

    try:
        import cupy as cp
        cp.cuda.Device(0).compute_capability
        info['cupy'] = True
    except Exception as e:
        log(f'CuPy недоступен: {e}')

    try:
        info['xgboost_gpu'] = xgb.build_info().get('USE_CUDA', 'OFF') == 'ON'
    except Exception:
        pass

    try:
        import lightgbm as lgb
        info['lightgbm_gpu'] = True
        globals()['lgb'] = lgb
        globals()['LGBMClassifier'] = lgb.LGBMClassifier
    except ImportError:
        log('LightGBM не установлен (опционально)')

    log(f'OS: {platform.platform()}')
    log(f'Polars: {pl.__version__}')
    log(f'XGBoost: {xgb.__version__}, CUDA build: {info["xgboost_gpu"]}')
    log(f'RAPIDS cuDF: {info["cudf"]}, cuML: {info["cuml"]}')
    return info


GPU = check_gpu_stack()
USE_RAPIDS = GPU['cudf'] and GPU['cuml']
USE_XGB_GPU = GPU['xgboost_gpu'] and GPU['nvidia_smi']

if USE_RAPIDS:
    import cudf
    import cupy as cp
    from cuml.ensemble import RandomForestClassifier as cuRFClassifier
    from cuml.preprocessing import LabelEncoder as cuLabelEncoder
    log('Режим: Polars → cuDF → GPU ML')
elif USE_XGB_GPU:
    log('Режим: Polars → pandas → XGBoost GPU (без RAPIDS)')
else:
    log('Режим: CPU fallback — установи CUDA/XGBoost GPU или RAPIDS в WSL2')

## 1. Вспомогательные функции

In [ ]:
def format_params(params, keys=None):
    if keys is not None:
        params = {k: v for k, v in params.items() if k in keys}
    return "\n".join([f"{k} = {v}" for k, v in params.items()])


def fmt(num):
    if num >= 1e9:
        return f"{num/1e9:.2f}B"
    if num >= 1e6:
        return f"{num/1e6:.2f}M"
    if num >= 1e3:
        return f"{num/1e3:.2f}K"
    return f"{num:.2f}"


def log_experiment(model_name, params, k_fold, feature_importance, features_info="",
                   log_file='experiments_log_v2_gpu.csv'):
    log_file = Path(log_file)
    result = {
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'Kaggle Score': '',
        'model': model_name,
        'K-Fold': k_fold,
        'feature_importance': feature_importance,
        'params': str(params),
        'features': features_info,
        'gpu_mode': 'RAPIDS' if USE_RAPIDS else ('XGB_GPU' if USE_XGB_GPU else 'CPU'),
        'os_full': platform.platform(),
    }
    if not log_file.exists():
        pl.DataFrame({k: pl.Series([], dtype=pl.Utf8) for k in result}).write_csv(log_file)
    df_log = pl.read_csv(log_file)
    df_log = pl.concat([df_log, pl.DataFrame([result])], how='diagonal')
    df_log.write_csv(log_file)
    print(f"Результат сохранён в {log_file}")
    display(df_log.sort('timestamp', descending=True))
    return df_log


def polars_to_gpu(df_pl: pl.DataFrame):
    """Polars → cuDF (GPU) или pandas (fallback)."""
    if USE_RAPIDS:
        return cudf.from_pandas(df_pl.to_pandas())
    return df_pl.to_pandas()


def gpu_to_polars(df_gpu):
    if USE_RAPIDS and hasattr(df_gpu, 'to_pandas'):
        return pl.from_pandas(df_gpu.to_pandas())
    return pl.from_pandas(df_gpu)

## 2. Загрузка данных (Polars)

In [ ]:
ID_COL = 'id'
TARGET_COL = 'class'
NUMERIC_COLS = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
CATEGORICAL_COLS = ['spectral_type', 'galaxy_population']

DATA_DIR = Path('data')
assert (DATA_DIR / 'train.csv').exists(), 'Положи train.csv и test.csv в папку data/'

t0 = time.perf_counter()
train_lf = pl.scan_csv(DATA_DIR / 'train.csv')
test_lf = pl.scan_csv(DATA_DIR / 'test.csv')

train_df = train_lf.collect()
test_df = test_lf.collect()
log(f'Загрузка Polars: {time.perf_counter() - t0:.3f} сек')

log(f'train: {train_df.shape}, test: {test_df.shape}')
display(train_df.head(5))
display(train_df.describe())

## 3. EDA (Polars)

In [ ]:
for col in CATEGORICAL_COLS:
    display(train_df[col].value_counts().sort('count', descending=True))

display(train_df[TARGET_COL].value_counts().sort('count', descending=True))

# Гистограммы числовых признаков
cols_to_plot = [c for c in NUMERIC_COLS]
ncols, nrows = 2, int(np.ceil(len(cols_to_plot) / 2))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 4))
axes = axes.flatten()

train_pd = train_df.to_pandas()
test_pd = test_df.to_pandas()

for i, col in enumerate(cols_to_plot):
    ax = axes[i]
    ax.hist(train_pd[col], bins=30, alpha=0.6, label='Train', density=True, color='skyblue')
    ax.hist(test_pd[col], bins=30, alpha=0.6, label='Test', density=True, color='lightgreen')
    ax.set_title(col)
    ax.legend()

for j in range(len(cols_to_plot), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

## 4. Feature Engineering (Polars — lazy pipeline)

Те же признаки, что в `solution_v2.ipynb`, но через Polars expressions.
Преимущество Polars: можно собрать весь пайплайн в `LazyFrame` и выполнить одним `.collect()`.

In [ ]:
# Константы для перевода экваториальных → галактических координат (J2000)
ALPHA_GP = np.radians(192.85948)
DELTA_GP = np.radians(27.12825)
L_NCP = np.radians(122.93192)


def add_features(lf: pl.LazyFrame) -> pl.LazyFrame:
    """Feature engineering на Polars (CPU, но очень быстрый)."""
    alpha_rad = pl.col('alpha').radians()
    delta_rad = pl.col('delta').radians()

    sin_b = (
        np.sin(DELTA_GP) * delta_rad.sin()
        + np.cos(DELTA_GP) * delta_rad.cos() * (alpha_rad - ALPHA_GP).cos()
    )
    b_rad = sin_b.arcsin()
    y = delta_rad.cos() * (alpha_rad - ALPHA_GP).sin()
    x = (
        np.cos(DELTA_GP) * delta_rad.sin()
        - np.sin(DELTA_GP) * delta_rad.cos() * (alpha_rad - ALPHA_GP).cos()
    )
    gal_l = ((x.arctan2(y) + L_NCP).degrees() % 360)
    gal_b = b_rad.degrees()

    # Светимость (упрощённая формула из v2)
    d_L = (299792.458 / 70) * pl.col('redshift') * (1 + (1 - (-0.55)) / 2 * pl.col('redshift'))
    mu = 5 * (d_L * 1e6).log10() - 5

    return (
        lf.with_columns([
            gal_l.alias('galactic_l'),
            gal_b.alias('galactic_b'),
        ])
        .with_columns([
            pl.col('galactic_l').radians().sin().alias('sin_gal_l'),
            pl.col('galactic_l').radians().cos().alias('cos_gal_l'),
            pl.col('galactic_b').radians().sin().alias('sin_gal_b'),
            pl.col('galactic_b').radians().cos().alias('cos_gal_b'),
            pl.col('alpha').radians().sin().alias('sin_alpha'),
            pl.col('alpha').radians().cos().alias('cos_alpha'),
            (pl.col('r') - mu).alias('abs_mag_r'),
            ((pl.col('u') - pl.col('g')) - (pl.col('g') - pl.col('r'))).alias('curvature_ugr'),
            ((pl.col('g') - pl.col('r')) - (pl.col('r') - pl.col('i'))).alias('curvature_gri'),
        ])
        .drop(['galactic_l', 'galactic_b', 'alpha'])
    )


t0 = time.perf_counter()
train_df = add_features(train_lf).collect()
test_df = add_features(test_lf).collect()
log(f'Feature engineering Polars: {time.perf_counter() - t0:.3f} сек')
display(train_df.head(3))

## 5. GPU EDA (cuDF)

Переносим данные на GPU и считаем корреляции через RAPIDS.
Если RAPIDS недоступен — используем Polars `.corr()`.

In [ ]:
numeric_for_corr = [c for c in train_df.columns if c not in [ID_COL, TARGET_COL] + CATEGORICAL_COLS]

if USE_RAPIDS:
    t0 = time.perf_counter()
    gdf = cudf.from_pandas(train_df.select(numeric_for_corr).to_pandas())
    corr_matrix = gdf.corr().to_pandas()
    log(f'Корреляция на GPU (cuDF): {time.perf_counter() - t0:.3f} сек')
    log(f'GPU memory: {gdf.memory_usage(deep=True).sum() / 1e6:.1f} MB')
else:
    t0 = time.perf_counter()
    corr_matrix = train_df.select(numeric_for_corr).to_pandas().corr()
    log(f'Корреляция Polars/pandas (CPU): {time.perf_counter() - t0:.3f} сек')

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='vlag', center=0, fmt='.2f', square=True)
plt.title('Correlation Matrix (GPU cuDF)' if USE_RAPIDS else 'Correlation Matrix (CPU)')
plt.tight_layout()
plt.show()

# Корреляция с target
le = LabelEncoder()
y_enc = le.fit_transform(train_df[TARGET_COL].to_numpy())
correlations = []
for col in numeric_for_corr:
    x = train_df[col].to_numpy()
    corr = np.corrcoef(x, y_enc)[0, 1]
    correlations.append((col, corr))
correlations.sort(key=lambda x: abs(x[1]), reverse=True)
print('Корреляция с target:')
for col, corr in correlations[:10]:
    print(f'  {col:20s}: {corr:+.4f}')

## 6. Подготовка данных для ML

In [ ]:
# Label encoding категориальных признаков
cat_encoders = {}
train_ml = train_df.clone()
test_ml = test_df.clone()

for col in CATEGORICAL_COLS:
    le_cat = LabelEncoder()
    all_vals = pl.concat([train_ml[col], test_ml[col]]).unique().to_numpy()
    le_cat.fit(all_vals)
    cat_encoders[col] = le_cat
    train_ml = train_ml.with_columns(pl.Series(col, le_cat.transform(train_ml[col].to_numpy())).cast(pl.Int32))
    test_ml = test_ml.with_columns(pl.Series(col, le_cat.transform(test_ml[col].to_numpy())).cast(pl.Int32))

feature_cols = [c for c in train_ml.columns if c not in [ID_COL, TARGET_COL]]
X_pl = train_ml.select(feature_cols)
y_pl = train_ml[TARGET_COL]
X_test_pl = test_ml.select(feature_cols)
test_ids = test_ml[ID_COL]

log(f'Признаков: {len(feature_cols)}')
display(feature_cols)

## 7. Модели на GPU

### 7.1 XGBoost GPU (основная модель — аналог LightGBM из v2)

XGBoost с `device='cuda'` работает и в WSL2, и нативно на Windows.

In [ ]:
KEY_PARAMS = [
    'n_estimators', 'learning_rate', 'max_depth',
    'subsample', 'colsample_bytree', 'min_child_weight',
]

XGB_PARAMS = {
    'n_estimators': 2000,
    'learning_rate': 0.03,
    'max_depth': 10,
    'min_child_weight': 1,
    'subsample': 0.9,
    'colsample_bytree': 0.9,
    'objective': 'multi:softprob',
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',
    'device': 'cuda' if USE_XGB_GPU else 'cpu',
    'random_state': 42,
    'verbosity': 0,
}

le_target = LabelEncoder()
X_np = X_pl.to_numpy().astype(np.float32)
y_np = le_target.fit_transform(y_pl.to_numpy())
X_test_np = X_test_pl.to_numpy().astype(np.float32)
class_names = le_target.classes_
n_classes = len(class_names)

log(f'XGBoost device: {XGB_PARAMS["device"]}')

In [ ]:
def run_xgb_cv(X, y, params, n_splits=5, random_state=42):
    """5-fold CV на XGBoost GPU."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_scores = []
    oof = np.zeros((len(y), n_classes), dtype=np.float32)

    t0 = time.perf_counter()
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
        log(f'Fold {fold}/{n_splits}...')
        model = XGBClassifier(**params)
        model.fit(
            X[tr_idx], y[tr_idx],
            eval_set=[(X[val_idx], y[val_idx])],
            verbose=500,
        )
        proba = model.predict_proba(X[val_idx])
        oof[val_idx] = proba
        pred = proba.argmax(axis=1)
        score = balanced_accuracy_score(y[val_idx], pred)
        fold_scores.append(score)
        log(f'  Fold {fold} BalAcc: {score:.4f}')

    elapsed = time.perf_counter() - t0
    oof_pred = oof.argmax(axis=1)
    oof_score = balanced_accuracy_score(y, oof_pred)

    results = {
        'Mean BalAcc': f'{np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}',
        'Min': f'{min(fold_scores):.4f}',
        'Max': f'{max(fold_scores):.4f}',
        'OOF BalAcc': f'{oof_score:.4f}',
        'Time (sec)': f'{elapsed:.1f}',
    }
    return results, fold_scores, oof


cv_results, fold_scores, oof_preds = run_xgb_cv(X_np, y_np, XGB_PARAMS)
log('CV результаты:')
for k, v in cv_results.items():
    log(f'  {k}: {v}')

### 7.2 cuML Random Forest (только при наличии RAPIDS)

Демонстрация нативного ML API RAPIDS — данные остаются на GPU.

In [ ]:
if USE_RAPIDS:
    log('cuML Random Forest — 5-fold CV на GPU')

    X_cudf = cudf.DataFrame(X_np, columns=feature_cols)
    y_cudf = cudf.Series(y_np)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    rf_scores = []

    t0 = time.perf_counter()
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_np, y_np), 1):
        rf = cuRFClassifier(
            n_estimators=500,
            max_depth=16,
            n_bins=128,
            random_state=42,
        )
        rf.fit(X_cudf.iloc[tr_idx], y_cudf.iloc[tr_idx])
        pred = rf.predict(X_cudf.iloc[val_idx]).to_numpy()
        score = balanced_accuracy_score(y_np[val_idx], pred)
        rf_scores.append(score)
        log(f'  cuML RF Fold {fold} BalAcc: {score:.4f}')

    log(f'cuML RF Mean: {np.mean(rf_scores):.4f} ± {np.std(rf_scores):.4f}, time: {time.perf_counter()-t0:.1f}s')
else:
    log('cuML пропущен — установи RAPIDS в WSL2 для этого блока')

### 7.3 LightGBM GPU (опционально, если установлен)

In [ ]:
if GPU.get('lightgbm_gpu') and GPU['nvidia_smi']:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier

    LGBM_PARAMS = {
        'n_estimators': 2000,
        'learning_rate': 0.03,
        'max_depth': 10,
        'num_leaves': 63,
        'min_child_samples': 10,
        'subsample': 0.9,
        'colsample_bytree': 0.9,
        'class_weight': 'balanced',
        'device': 'gpu',
        'random_state': 42,
        'verbose': -1,
    }

    # LightGBM хочет pandas + category dtype для категориальных
    X_lgb = train_ml.drop([ID_COL, TARGET_COL]).to_pandas()
    for col in CATEGORICAL_COLS:
        X_lgb[col] = X_lgb[col].astype('category')
    y_lgb = y_pl.to_numpy()

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    lgb_scores = []
    t0 = time.perf_counter()
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_lgb, y_lgb), 1):
        m = LGBMClassifier(**LGBM_PARAMS)
        m.fit(
            X_lgb.iloc[tr_idx], y_lgb[tr_idx],
            eval_set=[(X_lgb.iloc[val_idx], y_lgb[val_idx])],
            callbacks=[lgb.log_evaluation(500)],
        )
        pred = m.predict(X_lgb.iloc[val_idx])
        lgb_scores.append(balanced_accuracy_score(y_lgb[val_idx], pred))
        log(f'  LGBM GPU Fold {fold} BalAcc: {lgb_scores[-1]:.4f}')
    log(f'LGBM GPU Mean: {np.mean(lgb_scores):.4f} ± {np.std(lgb_scores):.4f}, time: {time.perf_counter()-t0:.1f}s')
else:
    log('LightGBM GPU пропущен')

## 8. Финальная модель и submission

In [ ]:
final_model = XGBClassifier(**XGB_PARAMS)
final_model.fit(X_np, y_np, verbose=500)

proba_test = final_model.predict_proba(X_test_np)
pred_idx = proba_test.argmax(axis=1)
predictions = class_names[pred_idx]

log(f'Предсказаний: {len(predictions)}')
display(pl.Series(predictions).value_counts().sort('count', descending=True))

# Feature importance
importances = final_model.feature_importances_
fi_df = (
    pl.DataFrame({'Feature': feature_cols, 'Importance': importances})
    .sort('Importance', descending=True)
)
print('Топ-15 признаков (XGBoost gain):')
display(fi_df.head(15))

plt.figure(figsize=(10, 8))
top = fi_df.head(20).to_pandas()
plt.barh(top['Feature'][::-1], top['Importance'][::-1])
plt.xlabel('Importance')
plt.title('Feature Importance (XGBoost GPU)')
plt.tight_layout()
plt.show()

fi_top = {row['Feature']: fmt(row['Importance']) for row in fi_df.head(20).iter_rows(named=True)}

log_experiment(
    model_name='XGBoost-GPU' if USE_XGB_GPU else 'XGBoost-CPU',
    params=format_params(XGB_PARAMS, keys=KEY_PARAMS + ['device', 'tree_method']),
    k_fold=format_params(cv_results),
    feature_importance=format_params(fi_top),
    features_info='Polars FE\n+ sin/cos galactic\n+ sin/cos alpha\n+ abs_mag_r\n+ curvature',
)

In [ ]:
submission = pl.DataFrame({
    ID_COL: test_ids,
    TARGET_COL: predictions,
})
submission.write_csv('submission_gpu.csv')
log('Сохранено: submission_gpu.csv')
display(submission.head())

## 9. Шпаргалка: что где живёт

| Операция | CPU (старый стек) | GPU (новый стек) |
|----------|-------------------|------------------|
| Загрузка CSV | `pd.read_csv` | `pl.scan_csv().collect()` |
| FE / трансформации | pandas/numpy | **Polars** expressions |
| corr / describe | pandas | **cuDF** (RAPIDS) |
| GBDT | LightGBM CPU | **XGBoost `device=cuda`** или **LGBM `device=gpu`** |
| Random Forest | sklearn | **cuML** `RandomForestClassifier` |
| Массивы | numpy | **CuPy** (если нужны кастомные kernel-операции) |

### Типичный паттерн миграции

```python
# 1. Polars для ETL
df = pl.scan_csv('data.csv').with_columns(...).collect()

# 2. GPU DataFrame для аналитики
gdf = cudf.from_pandas(df.to_pandas())
corr = gdf.corr()

# 3. ML — numpy/cudf → модель на GPU
model = XGBClassifier(device='cuda', tree_method='hist')
model.fit(X_np, y_np)
```

### Полезные ссылки
- [RAPIDS install guide](https://docs.rapids.ai/install)
- [Polars user guide](https://docs.pola.rs/)
- [XGBoost GPU support](https://xgboost.readthedocs.io/en/stable/gpu/index.html)